# V-JEPA 2 sur de la VRAIE vidéo (test conditions réelles)

On charge le **vrai world model vidéo de Meta** (V-JEPA 2, ViT-L) et on vérifie qu'il comprend de la vraie vidéo, puis on sonde sa représentation. Pas de jouet, pas d'entraînement from scratch — on part du foundation model, comme en production.

Modèle : https://huggingface.co/facebook/vjepa2-vitl-fpc16-256-ssv2

## 1. GPU + installer transformers / torchcodec

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers torchcodec av

## 2. Charger V-JEPA 2 (classifieur d'actions, fine-tuné SSv2)

In [ ]:
import numpy as np, torch
from transformers import AutoModelForVideoClassification, AutoVideoProcessor
repo='facebook/vjepa2-vitl-fpc16-256-ssv2'
proc=AutoVideoProcessor.from_pretrained(repo)
model=AutoModelForVideoClassification.from_pretrained(repo, device_map='auto', attn_implementation='sdpa').eval()
print('V-JEPA 2 chargé |', sum(p.numel() for p in model.parameters())/1e6, 'M params')

## 3. Le faire tourner sur de la VRAIE vidéo
Note : ce classifieur est fine-tuné sur **SSv2** (gestes main-objet), pas Kinetics — donc les labels sur ce clip ne colleront pas (espaces différents). Le vrai test = **ça tourne sur de la vraie vidéo sur Colab** et produit des sorties structurées + des features (cellule 4).

In [ ]:
from torchcodec.decoders import VideoDecoder
URLS={
  'tir à l\'arc':'https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/archery/-Qz25rXdMjE_000014_000024.mp4',
}
for nom,u in URLS.items():
    vr=VideoDecoder(u); n=getattr(model.config,'frames_per_clip',16)
    idx=np.linspace(0, vr.metadata.num_frames-1, n).astype(int)
    vid=vr.get_frames_at(indices=idx).data
    inp=proc(vid, return_tensors='pt').to(model.device)
    with torch.no_grad(): logits=model(**inp).logits[0]
    top=logits.softmax(-1).topk(5)
    print(f'\nvidéo réelle ({nom}) -> top-5 prédits par V-JEPA 2 :')
    for s,i in zip(top.values.tolist(), top.indices.tolist()):
        print(f'   {s:.2f}  {model.config.id2label[i]}')

## 4. Sonder la REPRÉSENTATION (le world model, pas le classifieur)
On extrait les features gelées du backbone V-JEPA 2 — c'est elles qu'on probe/finetune pour une tâche réelle (conduite, physique…).

In [ ]:
from transformers import AutoModel
bb=AutoModel.from_pretrained('facebook/vjepa2-vitl-fpc64-256', device_map='auto', attn_implementation='sdpa').eval()
vr=VideoDecoder(list(URLS.values())[0]); idx=np.linspace(0, vr.metadata.num_frames-1, 64).astype(int)
inp=proc(vr.get_frames_at(indices=idx).data, return_tensors='pt').to(bb.device)
with torch.no_grad(): feat=bb(**inp).last_hidden_state
print('features V-JEPA 2 :', tuple(feat.shape), '-> représentation spatio-temporelle réelle, à probe/finetune')